# Import Required Libraries

The following libraries are imported for data manipulation, visualization, preprocessing, machine learning model development, handling imbalanced data, and model evaluation.   

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV

from sklearn.preprocessing import LabelEncoder,StandardScaler

from sklearn.impute import KNNImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,precision_score,recall_score,f1_score,roc_curve,roc_auc_score

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler,SMOTE,ADASYN

# Load the Dataset

The dataset is loaded into a Pandas DataFrame to begin the data analysis and preprocessing steps.

In [3]:
df = pd.read_csv("../Dataset/Risk_Alert_Classifier_Dataset.csv")

df.head()

,customer_id,age,gender,region,employment_type,annual_income_inr,credit_score,credit_utilization_ratio,missed_payments_12m,avg_late_payment_days,monthly_transaction_count,monthly_spend_inr,cash_advance_count_6m,complaints_last_6m,failed_login_attempts_3m,account_tenure_months,last_transaction_date,debt_balance_inr,risk_status
0,500001,43.0,Female,NaN,Salaried,82242.0,NaN,0.120,1,2.2,39,33889.0,0,2,4,70,2025-09-26,87273,0
1,500002,29.0,Female,Central,Salaried,32769.0,647.0,0.337,1,1.5,11,10853.0,1,1,1,34,2025-11-24,20600,0
2,500003,36.0,Male,East,Salaried,39731.0,727.0,0.175,0,3.9,45,25519.0,2,1,1,74,2025-09-26,47565,0
3,500004,28.0,Male,North,Unemployed,38990.0,553.0,0.472,7,23.3,103,17806.0,1,2,6,72,2025-10-03,43803,1
4,500005,36.0,Female,East,Self-Employed,41043.0,732.0,0.418,1,9.8,95,27114.0,0,1,1,11,2025-10-26,12008,0


# Dataset Shape

The shape of the dataset represents the total number of rows and columns available for analysis.

In [4]:
print("Dataset Shape :", df.shape)

Dataset Shape : (4600, 19)


# Display Column Names

Before analyzing the data, it is useful to know all the available features present in the dataset.

In [5]:
df.columns

Index(['customer_id', 'age', 'gender', 'region', 'employment_type',
       'annual_income_inr', 'credit_score', 'credit_utilization_ratio',
       'missed_payments_12m', 'avg_late_payment_days',
       'monthly_transaction_count', 'monthly_spend_inr',
       'cash_advance_count_6m', 'complaints_last_6m',
       'failed_login_attempts_3m', 'account_tenure_months',
       'last_transaction_date', 'debt_balance_inr', 'risk_status'],
      dtype='object')

# Dataset Information

The `info()` function provides detailed information about the dataset, including:

- Number of non-null values
- Data types
- Memory usage

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                4600 non-null   int64  
 1   age                        4460 non-null   float64
 2   gender                     4600 non-null   object 
 3   region                     4498 non-null   object 
 4   employment_type            4456 non-null   object 
 5   annual_income_inr          4434 non-null   float64
 6   credit_score               4384 non-null   float64
 7   credit_utilization_ratio   4453 non-null   float64
 8   missed_payments_12m        4600 non-null   int64  
 9   avg_late_payment_days      4600 non-null   float64
 10  monthly_transaction_count  4600 non-null   int64  
 11  monthly_spend_inr          4471 non-null   float64
 12  cash_advance_count_6m      4600 non-null   int64  
 13  complaints_last_6m         4600 non-null   int64

# Statistical Summary

The `describe()` function provides statistical information for all numerical features.

It helps understand the distribution and spread of the data.

In [15]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,4600.0,502300.500000,1328.049949,500001.000,501150.750,502300.50,503450.250,504600.000
age,4460.0,36.360314,10.670375,18.000,28.000,36.00,44.000,75.000
annual_income_inr,4434.0,41753.518268,17740.750972,15000.000,28980.000,38932.50,51282.500,163002.000
credit_score,4384.0,677.784443,64.888787,405.000,638.000,682.00,721.250,850.000
credit_utilization_ratio,4453.0,0.394721,0.205771,0.002,0.232,0.37,0.531,0.978
missed_payments_12m,4600.0,0.924130,1.300018,0.000,0.000,1.00,1.000,10.000
avg_late_payment_days,4600.0,5.538696,5.624891,0.100,2.100,3.90,6.600,47.100
monthly_transaction_count,4600.0,65.030000,24.180762,5.000,49.000,65.00,81.000,153.000
monthly_spend_inr,4471.0,21511.273541,10887.272864,3769.000,13422.500,19317.00,27147.000,87389.000
cash_advance_count_6m,4600.0,0.709783,1.020507,0.000,0.000,0.00,1.000,7.000


# Identify Numerical and Categorical Features

Before performing Exploratory Data Analysis (EDA), it is important to separate numerical and categorical features.

This helps us choose appropriate visualization techniques and preprocessing methods for different types of data.

In [16]:
# Numerical Columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns

# Categorical Columns
categorical_columns = df.select_dtypes(include=['object']).columns

print("Numerical Columns:")
print(list(numerical_columns))

print("\nCategorical Columns:")
print(list(categorical_columns))

Numerical Columns:
['customer_id', 'age', 'annual_income_inr', 'credit_score', 'credit_utilization_ratio', 'missed_payments_12m', 'avg_late_payment_days', 'monthly_transaction_count', 'monthly_spend_inr', 'cash_advance_count_6m', 'complaints_last_6m', 'failed_login_attempts_3m', 'account_tenure_months', 'debt_balance_inr', 'risk_status']

Categorical Columns:
['gender', 'region', 'employment_type', 'last_transaction_date']


# Missing Value Analysis

Missing values can negatively affect machine learning models if not handled properly.

Before applying any imputation technique, we first identify the columns containing missing values.

In [23]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

credit_score                 216
annual_income_inr            166
credit_utilization_ratio     147
employment_type              144
age                          140
monthly_spend_inr            129
region                       102
customer_id                    0
complaints_last_6m             0
debt_balance_inr               0
last_transaction_date          0
account_tenure_months          0
failed_login_attempts_3m       0
avg_late_payment_days          0
cash_advance_count_6m          0
monthly_transaction_count      0
missed_payments_12m            0
gender                         0
risk_status                    0
dtype: int64

# Duplicate Value Analysis

Duplicate records may introduce bias during model training.

Therefore, we first check whether duplicate rows exist in the dataset.